# Etap 3 — Analiza + dostrajanie progów

Wyciśnięcie maksimum z modelu (Etap 2) bez ponownego treningu — przez **per-klasowe progi decyzyjne**.

1. Prawdopodobieństwa (sigmoid) na dev
2. Próg per klasa → maksymalizacja F1 przy precyzji ≥ `MIN_PRECISION` (anty-„krzyk wilka")
3. Porównanie micro/macro F1: próg 0.5 vs dostrojone
4. F1 per klasa — gdzie model jest mocny, gdzie ślepy
5. Zapis `thresholds.npy` dla bota

Progi strojone na **pełnym dev** (więcej pozytywów dla rzadkich klas), wynik raportowany na **polskim dev**.

In [ ]:
import sys
sys.modules["torchvision"] = None  # zapobiega konfliktowi VideoReader z datasets

from google.colab import drive
drive.mount("/content/drive")

!pip install -q -U transformers datasets scikit-learn

## 1. Konfiguracja + wczytanie modelu

In [ ]:
from pathlib import Path
import numpy as np
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Uczenie Maszynowe/Project")
DATA_DIR    = PROJECT_DIR / "Data"
PROC_DIR    = DATA_DIR / "processed"
MODELS_DIR  = PROJECT_DIR / "Models"

MODEL_DIR    = MODELS_DIR / "xlm-roberta-large_semeval_pl"   # z Etapu 2
DATASET_PATH = PROC_DIR / "semeval_multilang"
MAX_LENGTH   = 256
TARGET_LANG  = "po"

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model     = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
device    = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

LABELS = (MODEL_DIR / "labels.txt").read_text(encoding="utf-8").splitlines()
NUM_LABELS = len(LABELS)
assert NUM_LABELS == 23
print("Model wczytany na:", device)


## 2. Dane dev (pełny wielojęzyczny + polski)

In [ ]:
from datasets import load_from_disk

ds  = load_from_disk(str(DATASET_PATH))
dev = ds["validation"]
dev_pl = dev.filter(lambda x: x["lang"] == TARGET_LANG)

# teksty + złote etykiety jako macierze
texts_all = dev["text"];      Y_all = np.array(dev["labels"], dtype=int)
texts_pl  = dev_pl["text"];   Y_pl  = np.array(dev_pl["labels"], dtype=int)

print(f"dev pełny: {len(texts_all)} | dev PL: {len(texts_pl)}")


## 3. Inference — prawdopodobieństwa (sigmoid)

Ten sam wzorzec inferencji trafia do bota.

In [ ]:
@torch.no_grad()
def predict_probs(texts, batch_size=32):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            list(texts[i:i+batch_size]),
            truncation=True, max_length=MAX_LENGTH,
            padding=True, return_tensors="pt",
        ).to(device)
        logits = model(**enc).logits
        out.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(out)

P_all = predict_probs(texts_all)   # (n_all, 23)
P_pl  = predict_probs(texts_pl)    # (n_pl, 23)
print("prawdopodobieństwa policzone:", P_all.shape, P_pl.shape)


## 4. Dostrajanie progów per klasa

Dla każdej klasy: najwyższy F1 przy precyzji ≥ `MIN_PRECISION`. Klasy bez progu spełniającego warunek są wyłączane (próg → 0.95).

In [ ]:
from sklearn.metrics import f1_score, precision_score

MIN_PRECISION = 0.30   # minimalna precyzja, by zaakceptować próg (anty-"krzyk wilka")

def tune_thresholds(probs, gold, grid=np.arange(0.05, 0.96, 0.01), min_precision=MIN_PRECISION):
    thr = np.full(probs.shape[1], 0.5, dtype=np.float32)
    suppressed = []
    for j in range(probs.shape[1]):
        if gold[:, j].sum() == 0:
            continue  # brak pozytywów -> zostaw 0.5
        best_f1, best_t = -1.0, None
        for t in grid:
            pred = (probs[:, j] >= t).astype(int)
            if pred.sum() == 0:
                continue
            if precision_score(gold[:, j], pred, zero_division=0) < min_precision:
                continue  # za niska precyzja -> odrzuć ten próg
            f1 = f1_score(gold[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        if best_t is None:
            # żaden próg nie osiągnął wymaganej precyzji -> klasa wyłączona (rzadko strzela)
            best_t = float(grid[-1])
            suppressed.append(LABELS[j])
        thr[j] = best_t
    return thr, suppressed

thresholds, suppressed = tune_thresholds(P_all, Y_all)

print(f"Progi per klasa (posortowane), MIN_PRECISION={MIN_PRECISION}:")
for l, t in sorted(zip(LABELS, thresholds), key=lambda x: x[1]):
    flaga = "   <- wyłączona (za niska precyzja)" if l in suppressed else ""
    print(f"  {t:.2f}  {l}{flaga}")
if suppressed:
    print(f"\nWyłączone klasy ({len(suppressed)}): {', '.join(suppressed)}")


## 5. Wynik: próg 0.5 vs progi dostrojone

In [ ]:
def ocena(probs, gold, thr):
    preds = (probs >= thr).astype(int)
    return (f1_score(gold, preds, average="micro", zero_division=0),
            f1_score(gold, preds, average="macro", zero_division=0))

p05 = np.full(NUM_LABELS, 0.5, dtype=np.float32)

mi_pl_05,  ma_pl_05  = ocena(P_pl,  Y_pl,  p05)
mi_pl_tn,  ma_pl_tn  = ocena(P_pl,  Y_pl,  thresholds)
mi_all_05, ma_all_05 = ocena(P_all, Y_all, p05)
mi_all_tn, ma_all_tn = ocena(P_all, Y_all, thresholds)

print(f"{'':<22}{'micro-F1':>10}{'macro-F1':>10}")
print(f"{'PL dev  @ 0.5':<22}{mi_pl_05:>10.4f}{ma_pl_05:>10.4f}")
print(f"{'PL dev  @ tuned':<22}{mi_pl_tn:>10.4f}{ma_pl_tn:>10.4f}")
print(f"{'  -> przyrost':<22}{mi_pl_tn-mi_pl_05:>+10.4f}{ma_pl_tn-ma_pl_05:>+10.4f}")
print()
print(f"{'ALL dev @ 0.5':<22}{mi_all_05:>10.4f}{ma_all_05:>10.4f}")
print(f"{'ALL dev @ tuned':<22}{mi_all_tn:>10.4f}{ma_all_tn:>10.4f}")


## 6. F1 per klasa (pełny dev, progi dostrojone)

In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

preds_05 = (P_all >= p05).astype(int)
preds_tn = (P_all >= thresholds).astype(int)

f1_05 = f1_score(Y_all, preds_05, average=None, zero_division=0)
prec, rec, f1_tn, support = precision_recall_fscore_support(
    Y_all, preds_tn, average=None, zero_division=0
)

tabela = pd.DataFrame({
    "technika":  LABELS,
    "support":   support,
    "prog":      thresholds.round(2),
    "F1@0.5":    f1_05.round(3),
    "F1@tuned":  f1_tn.round(3),
    "precyzja":  prec.round(3),
    "recall":    rec.round(3),
}).sort_values("F1@tuned", ascending=False).reset_index(drop=True)

pd.set_option("display.max_rows", None)
print(tabela.to_string(index=False))

slepe = tabela[tabela["F1@tuned"] == 0]["technika"].tolist()
if slepe:
    print("\nKlasy z F1=0 (model ich nie łapie):", ", ".join(slepe))


## 7. Zapis progów dla bota

In [ ]:
np.save(MODEL_DIR / "thresholds.npy", thresholds)   # obok modelu
np.save(PROC_DIR / "thresholds.npy", thresholds)    # kopia w processed
print("Zapisano thresholds.npy do:")
print(" -", MODEL_DIR / "thresholds.npy")
print(" -", PROC_DIR / "thresholds.npy")


---
## Co dalej — bot Telegram

Do produkcji potrzebne są **3 artefakty**: model + tokenizer, `labels.txt`, `thresholds.npy`.

Bot: tekst → podział na akapity → `predict_probs` → `thresholds` → techniki z prawdopodobieństwem.